# 15. Saving and Loading Arrays

NumPy has its own binary format (`.npy` / `.npz`) and can also read and write plain text like CSV.

This notebook writes everything into a temporary folder so it doesn't leave files lying around.

In [1]:
import numpy as np
import tempfile, os

tmp = tempfile.mkdtemp()
path = lambda name: os.path.join(tmp, name)

rng = np.random.default_rng(0)
arr = rng.random((3, 4))

## `.npy`: a single array

Binary, fast, and keeps the exact dtype and shape. This is what you should use for saving arrays between your own Python sessions.

In [2]:
np.save(path('arr.npy'), arr)
loaded = np.load(path('arr.npy'))
print(np.array_equal(arr, loaded), loaded.dtype, loaded.shape)

True float64 (3, 4)


## `.npz`: several arrays in one file

`savez` stores multiple arrays under names. `savez_compressed` does the same with zip compression, which helps a lot for data with repetition.

In [3]:
labels = np.array([0, 1, 1])
np.savez(path('dataset.npz'), features=arr, labels=labels)

with np.load(path('dataset.npz')) as f:
    print(f.files)
    print(f['labels'])

['features', 'labels']
[0 1 1]


In [4]:
sparse_ish = np.zeros((1000, 1000))
np.savez(path('plain.npz'), a=sparse_ish)
np.savez_compressed(path('small.npz'), a=sparse_ish)
for name in ['plain.npz', 'small.npz']:
    print(name, os.path.getsize(path(name)) // 1024, 'KB')

plain.npz 7812 KB
small.npz 7 KB


## Memory mapping big files

For arrays too big to fit in RAM, `mmap_mode` lets you open a `.npy` file and only read the parts you touch:

In [5]:
np.save(path('big.npy'), np.arange(10_000_000, dtype=np.int32))
mm = np.load(path('big.npy'), mmap_mode='r')
print(type(mm).__name__, mm[5_000_000:5_000_005])

memmap [5000000 5000001 5000002 5000003 5000004]


## Text files

Text is slower and bigger, but human-readable and works with anything (Excel, other languages, ...).

### Writing with `savetxt`

In [6]:
np.savetxt(path('arr.csv'), arr, delimiter=',', fmt='%.4f',
           header='a,b,c,d', comments='')
print(open(path('arr.csv')).read())

a,b,c,d
0.6370,0.2698,0.0410,0.0165
0.8133,0.9128,0.6066,0.7295
0.5436,0.9351,0.8159,0.0027



### Reading with `loadtxt`

`loadtxt` is fine for clean, purely numeric files:

In [7]:
back = np.loadtxt(path('arr.csv'), delimiter=',', skiprows=1)
back

array([[0.637 , 0.2698, 0.041 , 0.0165],
       [0.8133, 0.9128, 0.6066, 0.7295],
       [0.5436, 0.9351, 0.8159, 0.0027]])

In [8]:
# pick specific columns
np.loadtxt(path('arr.csv'), delimiter=',', skiprows=1, usecols=(0, 2))

array([[0.637 , 0.041 ],
       [0.8133, 0.6066],
       [0.5436, 0.8159]])

### Messier files with `genfromtxt`

`genfromtxt` handles missing values and can use the header row as field names:

In [9]:
csv = '''name,age,score
alice,30,88.5
bob,,92.0
carol,25,
'''
with open(path('people.csv'), 'w') as f:
    f.write(csv)

people = np.genfromtxt(path('people.csv'), delimiter=',', names=True,
                       dtype=None, encoding='utf-8')
print(people)
print(people['age'])
print(people['score'])

[('alice', 30, 88.5) ('bob', -1, 92. ) ('carol', 25,  nan)]
[30 -1 25]
[88.5 92.   nan]


Missing numbers come back as `-1` for ints and `nan` for floats by default (you can change this with `filling_values`). The result is a **structured array**, which you index by column name. For anything more complicated than this, use pandas (`pd.read_csv`) instead.

## Which format should I use?

| Situation | Use |
|---|---|
| Saving work to reload in Python later | `np.save` / `np.load` |
| Several related arrays | `np.savez` (or `savez_compressed`) |
| Data bigger than memory | `.npy` + `mmap_mode` |
| Sharing with people or other tools | `savetxt` / CSV |
| Mixed column types, messy CSV | pandas |

One more thing: `np.load` won't load files containing Python objects unless you pass `allow_pickle=True`. Only do that for files you trust, since unpickling can run arbitrary code.

In [10]:
import shutil
shutil.rmtree(tmp)